# PIPELINE - Limpieza ENAHO

## Introducción: El PIPELINE de Limpieza y Agregación de ENAHO

Este notebook implementa un pipeline integral para la limpieza, procesamiento y agregación de datos de la **Encuesta Nacional de Hogares (ENAHO)**, una fuente crucial de información socioeconómica en el Perú. El objetivo principal es transformar los microdatos de la ENAHO en indicadores distritales significativos, que puedan ser utilizados para el análisis de políticas públicas y la identificación de patrones a nivel geográfico.

El pipeline se estructura en las siguientes fases:

1.  **Carga y Selección de Módulos**: Se cargan los módulos relevantes de la ENAHO (Hogar, Miembros del Hogar, Educación, Salud) y se realiza una selección inicial de variables.
2.  **Limpieza y Recodificación por Módulo**: Cada módulo pasa por un proceso de limpieza individual, que incluye:
    *   Diagnóstico de tipos de variables.
    *   Recodificación de variables categóricas y binarias a formatos numéricos (0/1 o escalas discretas).
    *   Renombramiento de columnas para una mayor legibilidad y estandarización.
3.  **Fusión a Nivel Individual**: Los módulos procesados se fusionan para crear un dataset único a nivel de individuo, que contendrá todas las variables relevantes por persona.
4.  **Agregación Ponderada por Distrito**: Se calculan medias y proporciones ponderadas de las variables a nivel distrital (UBIGEO), utilizando los factores de expansión muestral provistos por la ENAHO.
5.  **Filtro de Confiabilidad Muestral**: Se aplica un filtro para asegurar que solo se retengan los distritos con un tamaño muestral mínimo, garantizando la robustez de los indicadores.

Este proceso es fundamental para preparar los datos de la ENAHO para análisis posteriores, como la construcción de índices o modelos predictivos a nivel subnacional.

In [3]:
import pandas as pd
import matplotlib.pyplot as plt

### Preparación del Entorno

Antes de iniciar con el procesamiento de los datos, importamos las librerías necesarias para la manipulación y análisis de DataFrames, así como para la visualización de datos.

## Módulo 100: hogar

### Módulo 100: Características de la Vivienda y el Hogar

El Módulo 100 de la ENAHO contiene información detallada sobre las características de la vivienda y del hogar, incluyendo tipo de vivienda, acceso a servicios básicos, materiales de construcción y Necesidades Básicas Insatisfechas (NBI). Esta sección se encarga de cargar este módulo, inspeccionar su estructura inicial y seleccionar las variables pertinentes para el análisis.

In [4]:
import sys
!{sys.executable} -m pip install pyreadstat
# carga del módulo 100
enaho_100 = pd.read_spss('../data/raw/Enaho01-2025-100.sav')


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


#### Carga de Datos del Módulo 100

Cargamos el archivo `.sav` correspondiente al Módulo 100 (`Enaho01-2025-100.sav`) utilizando `pandas` y el paquete `pyreadstat` (asegurándonos de que esté instalado).

In [5]:
enaho_100.shape

(44599, 336)

#### Dimensiones Iniciales del Módulo 100

Verificamos la cantidad de filas (registros) y columnas (variables) del DataFrame recién cargado para tener una primera idea de su tamaño.

In [6]:
# principales características del módulo 100
print('Módulo 100 (sin procesar) - Características generales de la vivienda y del hogar:')
print(f'Número de registros: {enaho_100.shape[0]}')
print(f'Número de variables: {enaho_100.shape[1]}')

Módulo 100 (sin procesar) - Características generales de la vivienda y del hogar:
Número de registros: 44599
Número de variables: 336


#### Selección y Filtrado de Variables del Módulo 100

Aquí definimos una lista de variables de interés del Módulo 100, agrupadas por categorías conceptuales (identificación, infraestructura, conectividad, servicios básicos, NBI). Luego, filtramos el DataFrame `enaho_100` para incluir únicamente estas variables. Algunas variables relacionadas con materiales de vivienda (`P102`, `P103`, `P103A`, `P104`, `P104A`) y procedencia de agua (`P110`, `T110`) están comentadas porque fueron excluidas en decisiones previas de modelado o para evitar redundancia.

### Selección de variables de interés

In [7]:
variables_modulo_100 = [
    # --- IDENTIFICACIÓN Y CONTEXTO GEOGRÁFICO ---
    "AÑO",           # Año de la encuesta
    "MES",           # Mes de la encuesta
    "UBIGEO",        # Ubicación geográfica (Distrito)
    "CONGLOME",      # Conglomerado
    "VIVIENDA",      # Número de vivienda
    "HOGAR",         # Número de hogar
    "DOMINIO",       # Dominio Geográfico (Costa, Sierra, Selva, etc.)
    "ESTRATO",       # Estrato Geográfico / Tamaño de la población
    "FACTOR07",      # Factor de expansión anual

    # --- INFRAESTRUCTURA Y ENTORNO DE LA VIVIENDA ---
    "P101",          # Tipo de vivienda
    #"P102",          # Material predominante en las paredes exteriores
    #"P103",          # Material predominante en los pisos
    #"P103A",         # Material predominante en los techos
    #"P104",          # Cantidad de habitaciones en total en la vivienda
    #"P104A",         # Cantidad de habitaciones usadas exclusivamente para dormir

    # --- CONECTIVIDAD Y CANALES DE COMUNICACIÓN EXTERNA ---
    "P1142",         # El hogar tiene: Teléfono Celular
    "P1144",         # El hogar tiene: Conexión a Internet (Fijo/Móvil)

    # --- SERVICIOS BÁSICOS (ESTRESORES DOMÉSTICOS) ---
    #"P110",          # Procedencia principal del agua que utilizan en el hogar
    #"T110",          # (Recodificado) Procedencia del agua que utilizan en el hogar
    "P110C",         # ¿El hogar tiene acceso al servicio de agua todos los días de la semana?
    # "P110C1",      # (excluida a pedido) Horas de agua al día si tiene todos los días
    # "P110C3",      # (excluida a pedido) Horas de agua al día si no tiene todos los días
    "P1121",         # Tipo de alumbrado del hogar: Electricidad

    # --- BLOQUE COMPLETO DE NECESIDADES BÁSICAS INSATISFECHAS (NBI) ---
    "NBI1",          # Vivienda inadecuada (NBI 1)
    "NBI2",          # Vivienda con hacinamiento (NBI 2)
    "NBI3",          # Hogares con vivienda sin servicios higiénicos (NBI 3)
    "NBI4",          # Hogares con niños que no asisten a la escuela (NBI 4)
    "NBI5"           # Hogares con alta dependencia económica (NBI 5)
]

enaho_100 = enaho_100[variables_modulo_100]

print(f'Módulo 100 (filtrado) - Características generales de la vivienda y del hogar:')
print(f'Número de registros: {enaho_100.shape[0]}')
print(f'Número de variables: {enaho_100.shape[1]}')

Módulo 100 (filtrado) - Características generales de la vivienda y del hogar:
Número de registros: 44599
Número de variables: 19


#### Diagnóstico de Columnas del Módulo 100

Esta sección realiza una auditoría automática de los tipos de variables presentes en el DataFrame `enaho_100`. Clasifica las columnas en binarias, categóricas o numéricas, y también identifica columnas administrativas o vacías. Esto es útil para entender la naturaleza de los datos y planificar las recodificaciones necesarias. Las columnas clave de identificación (`AÑO`, `MES`, `UBIGEO`, etc.) se tratan como administrativas y se omiten del análisis de tipos de datos para recodificación.

### Diagnostico de columnas

#### Conversión y Recodificación de Variables Binarias

En esta etapa, se recodifican las variables binarias del Módulo 100 a un formato numérico (0 y 1). Esto incluye variables relacionadas con el acceso a tecnologías y servicios (teléfono celular, internet, electricidad, agua todos los días) y las variables de Necesidades Básicas Insatisfechas (NBI). Se crea una copia (`enaho_100_recoded`) para preservar el DataFrame original.

In [8]:


# =====================================================================
# CELDA 2 (OPCIONAL / DIAGNÓSTICO): Auditoría de tipos de variable
# =====================================================================
# Este bloque NO modifica enaho_100, solo te ayuda a inspeccionar cuántas
# columnas son binarias, categóricas o numéricas. Las listas que genera
# (binary_cols, categorical_cols, numeric_cols, skipped_cols) NO se usan
# en las celdas siguientes, porque ahí la recodificación se hace a mano
# columna por columna. Puedes saltarte esta celda sin afectar el resto
# del notebook; la dejo solo como chequeo de auditoría.

total_columns_shape = enaho_100.shape[1]

binary_cols = []
categorical_cols = []
numeric_cols = []
skipped_cols = []

administrative_cols = ['AÑO', 'MES','UBIGEO', 'CONGLOME', 'VIVIENDA', 'HOGAR', 'FACTOR07']

for col in enaho_100.columns:
    if col in administrative_cols:
        skipped_cols.append(col)
        continue

    unique_vals = enaho_100[col].dropna().unique()
    num_unique = len(unique_vals)

    if num_unique == 0:
        skipped_cols.append(col)
    elif num_unique == 1:
        skipped_cols.append(col)
    elif num_unique == 2:
        binary_cols.append(col)
    elif enaho_100[col].dtype in ['object', 'category'] or num_unique < 15:
        categorical_cols.append(col)
    else:
        numeric_cols.append(col)

total_accounted_for = len(binary_cols) + len(categorical_cols) + len(numeric_cols) + len(skipped_cols)
all_accounted_columns = binary_cols + categorical_cols + numeric_cols + skipped_cols
missing_columns = [col for col in enaho_100.columns if col not in all_accounted_columns]

print(f"Total de columnas previstas: {total_columns_shape}")
print(f"Total de columnas encontradas:    {total_accounted_for}")
print("-" * 50)
print(f"Columnas binarias:      {len(binary_cols)}")
print(f"Columnas categóricas: {len(categorical_cols)}")
print(f"Columnas numéricas:     {len(numeric_cols)}")
print(f"Administrativas/Vacías: {len(skipped_cols)}")
print("-" * 50)

if len(missing_columns) > 0:
    print(f"⚠️ WARNING: There are {len(missing_columns)} columns completely missing from your lists!")
    print(f"The missing column names are: {missing_columns}")
else:
    print("La cantidad de columnas coincide. No hay columnas sin clasificar.")



Total de columnas previstas: 19
Total de columnas encontradas:    19
--------------------------------------------------
Columnas binarias:      9
Columnas categóricas: 3
Columnas numéricas:     0
Administrativas/Vacías: 7
--------------------------------------------------
La cantidad de columnas coincide. No hay columnas sin clasificar.


### Conversión de valores en columnas binarias

#### Aplicación de Recodificaciones Binarias

Las variables se mapean para que los valores afirmativos (ej. 'Tiene Teléfono Celular') se conviertan en 1.0 y los negativos ('No tiene' o 'Pase') en 0.0. Esto facilita el análisis cuantitativo y la interpretación de los indicadores. Se verifica la existencia de la columna antes de aplicar el mapeo para evitar errores.

In [9]:

# =====================================================================
# CELDA 3: Recodificación binaria — servicios y NBI
# =====================================================================
# A partir de aquí trabajamos sobre una copia explícita, enaho_100_recoded,
# que irá acumulando TODAS las recodificaciones (esta celda y la siguiente).

enaho_100_recoded = enaho_100.copy()

# 1. Tecnología y servicios: 1.0 = tiene el servicio, 0.0 = no lo tiene ('Pase')
tech_utilities_maps = {
    'P1142': {'Telefono Celular': 1.0, 'Pase': 0.0},
    'P1144': {'Conexion a Internet': 1.0, 'Pase': 0.0},
    'P1121': {'Electricidad': 1.0, 'Pase': 0.0},
    'P110C': {'Si': 1.0, 'No': 0.0}
}

for col, mapping in tech_utilities_maps.items():
    if col in enaho_100_recoded.columns:
        enaho_100_recoded[col] = enaho_100_recoded[col].map(mapping).astype(float)

# 2. Necesidades Básicas Insatisfechas (NBI)
# 1.0 = presenta la carencia/vulnerabilidad, 0.0 = situación adecuada.
nbi_maps = {
    'NBI1': {'Vivienda inadecuada': 1.0, 'Vivienda adecuada': 0.0},
    'NBI2': {'Vivienda con hacinamiento': 1.0, 'Vivienda sin hacinamiento': 0.0},
    'NBI3': {'Hogares con vivienda sin servicios higienicos': 1.0, 'Hogares con vivienda con servicios higienicos': 0.0},
    'NBI4': {'Hogares con niños que no asisten a la escuela': 1.0, 'Hogares con niños que asisten a la escuela': 0.0},
    'NBI5': {'Hogares con alta dependencia economica': 1.0, 'Hogares sin alta dependencia economica': 0.0}
}

for col, mapping in nbi_maps.items():
    if col in enaho_100_recoded.columns:
        enaho_100_recoded[col] = enaho_100_recoded[col].map(mapping).astype(float)

print("✅ Recodificación binaria de servicios y NBI completa.")



✅ Recodificación binaria de servicios y NBI completa.


#### Recodificación de Variables Categóricas y Numéricas Discretas

Esta sección se ocupa de convertir variables categóricas y numéricas discretas a un formato estandarizado. La variable `MES` se convierte a un tipo numérico flotante. La variable `P101` (Tipo de vivienda) se recodifica para identificar viviendas consideradas "adecuadas" (Casa independiente, Departamento en edificio) con 1.0 y el resto con 0.0. Otras variables de materiales de vivienda (`P102`, `P103`, `P103A`) y procedencia de agua (`P110`) están comentadas, siguiendo la decisión de exclusión previa.

### Conversión de valores en columnas categóricas


#### Aplicación de Recodificaciones Categóricas y Numéricas

Los valores de `MES` se convierten a formato numérico. Para `P101`, se asigna 1.0 a tipos de vivienda específicos y 0.0 a otros. Las variables relacionadas con materiales de pared, pisos, techos y procedencia de agua están comentadas, lo que significa que no se están recodificando en este paso y, por lo tanto, no se incluirán en el análisis posterior con sus nombres recodificados.

In [10]:

# =====================================================================
# CELDA 4: Recodificación numérica discreta y categórica de vivienda
# =====================================================================
# IMPORTANTE: seguimos sobre enaho_100_recoded (resultado de la Celda 3),
# no sobre una copia nueva del original, para no perder lo ya recodificado.


enaho_100_recoded['MES'] = pd.to_numeric(enaho_100_recoded['MES'], errors='coerce').astype(float)
# 2. Variables categóricas de vivienda (1.0 = adecuado/noble, 0.0 = vulnerable/otro)

# P101: Tipo de vivienda
enaho_100_recoded["P101"] = (
    enaho_100_recoded["P101"]
    .isin(["Casa independiente", "Departamento en edificio"])
    .astype(float)
)

"""
# 1. Variables numéricas discretas
numeric_scale_cols = ["P104", "P104A"]
for col in numeric_scale_cols:
    enaho_100_recoded[col] = pd.to_numeric(enaho_100_recoded[col], errors="coerce").astype(float)



# P102: Material de paredes
enaho_100_recoded["P102"] = (
    enaho_100_recoded["P102"]
    .isin(["Ladrillo o bloque de cemento", "Piedra o sillar con cal o cemento"])
    .astype(float)
)

# P103: Material de pisos
enaho_100_recoded["P103"] = (
    enaho_100_recoded["P103"]
    .isin(
        [
            "Cemento",
            "Losetas, terrazos o similares",
            "Laminas asfalticas, vinilicos o similares",
            "Parquet o madera pulida",
        ]
    )
    .astype(float)
)

# P103A: Material de techos
enaho_100_recoded["P103A"] = (
    enaho_100_recoded["P103A"].isin(["Concreto armado", "Tejas"]).astype(float)
)

# P110: Infraestructura de agua
enaho_100_recoded["P110"] = (
    enaho_100_recoded["P110"]
    .isin(
        [
            "Red publica, dentro de la vivienda",
            "Red publica, fuera de la vivienda pero dentro del edificio",
        ]
    )
    .astype(float)
)

print("✅ Recodificación de variables categóricas y numéricas completada.")
print(f"Forma final del dataframe: {enaho_100_recoded.shape}")

"""

'\n# 1. Variables numéricas discretas\nnumeric_scale_cols = ["P104", "P104A"]\nfor col in numeric_scale_cols:\n    enaho_100_recoded[col] = pd.to_numeric(enaho_100_recoded[col], errors="coerce").astype(float)\n\n\n\n# P102: Material de paredes\nenaho_100_recoded["P102"] = (\n    enaho_100_recoded["P102"]\n    .isin(["Ladrillo o bloque de cemento", "Piedra o sillar con cal o cemento"])\n    .astype(float)\n)\n\n# P103: Material de pisos\nenaho_100_recoded["P103"] = (\n    enaho_100_recoded["P103"]\n    .isin(\n        [\n            "Cemento",\n            "Losetas, terrazos o similares",\n            "Laminas asfalticas, vinilicos o similares",\n            "Parquet o madera pulida",\n        ]\n    )\n    .astype(float)\n)\n\n# P103A: Material de techos\nenaho_100_recoded["P103A"] = (\n    enaho_100_recoded["P103A"].isin(["Concreto armado", "Tejas"]).astype(float)\n)\n\n# P110: Infraestructura de agua\nenaho_100_recoded["P110"] = (\n    enaho_100_recoded["P110"]\n    .isin(\n        [

#### Renombramiento de Columnas del Módulo 100

Para facilitar el trabajo con el DataFrame y mejorar la legibilidad, se renombran las columnas del Módulo 100 utilizando nombres más cortos, descriptivos y en minúsculas. Se utiliza un diccionario de mapeo para aplicar los nuevos nombres, asegurándose de que solo se renombren las columnas que realmente existen en el DataFrame `enaho_100_recoded`. El resultado es `enaho_100_final`.

### Renombramiento de variables

#### Aplicación del Renombramiento de Columnas

Las columnas del DataFrame `enaho_100_recoded` se renombran según el `diccionario_renombrar`. Esto estandariza los nombres de las variables, preparándolas para la fusión con otros módulos. Se muestra la lista de las nuevas columnas para verificar la aplicación correcta.

In [11]:
import pandas as pd

# Diccionario de mapeo con nombres cortos en minúsculas y español
diccionario_renombrar = {
    # --- IDENTIFICACIÓN Y CONTEXTO GEOGRÁFICO ---
    "AÑO": "anio",
    "MES": "mes",
    "UBIGEO": "ubigeo",
    "CONGLOME": "conglome",
    "VIVIENDA": "vivienda",
    "HOGAR": "hogar",
    "DOMINIO": "dominio",
    "ESTRATO": "estrato",
    "FACTOR07": "factor",
    # --- INFRAESTRUCTURA Y ENTORNO DE LA VIVIENDA ---
    "P101": "viv_tipo",
    "P102": "viv_paredes",
    "P103": "viv_pisos",
    "P103A": "viv_techos",
    "P104": "viv_hab_total",
    "P104A": "viv_hab_dormir",
    # --- CONECTIVIDAD Y CANALES DE COMUNICACIÓN EXTERNA ---
    "P1142": "tic_celular",
    "P1144": "tic_internet",
    # --- SERVICIOS BÁSICOS (ESTRESORES DOMÉSTICOS) ---
    "P110": "agua_procedencia",
    "T110": "agua_procedencia_recod",
    "P110C": "agua_todos_dias",
    "P110C1": "agua_horas_si_todos_dias",
    "P110C3": "agua_horas_no_todos_dias",
    "P1121": "serv_electricidad",
    # --- BLOQUE COMPLETO DE NECESIDADES BÁSICAS INSATISFECHAS (NBI) ---
    "NBI1": "nbi_vivienda",
    "NBI2": "nbi_hacinamiento",
    "NBI3": "nbi_sshh",
    "NBI4": "nbi_educacion",
    "NBI5": "nbi_dependencia",
}

# Aplicar el renombrado solo a las columnas que existan en tu DataFrame
enaho_100_final = enaho_100_recoded.rename(columns=diccionario_renombrar)

print("✅ Columnas renombradas exitosamente.")
print("Nuevas columnas en el dataset:\n", list(enaho_100_final.columns))

✅ Columnas renombradas exitosamente.
Nuevas columnas en el dataset:
 ['anio', 'mes', 'ubigeo', 'conglome', 'vivienda', 'hogar', 'dominio', 'estrato', 'factor', 'viv_tipo', 'tic_celular', 'tic_internet', 'agua_todos_dias', 'serv_electricidad', 'nbi_vivienda', 'nbi_hacinamiento', 'nbi_sshh', 'nbi_educacion', 'nbi_dependencia']


### Módulo 200: Miembros del Hogar

El Módulo 200 de la ENAHO proporciona información a nivel individual sobre los miembros del hogar, incluyendo datos demográficos como edad, sexo, relación de parentesco, estado civil y factor de expansión poblacional. Esta sección carga, selecciona y limpia las variables de este módulo.

## Módulo 200: Miembros del hogar

#### Carga de Datos del Módulo 200

Cargamos el archivo `.sav` correspondiente al Módulo 200 (`Enaho01-2025-200.sav`).

In [12]:
enaho_200 = pd.read_spss('../data/raw/Enaho01-2025-200.sav')

#### Selección de Variables del Módulo 200

Definimos las variables de interés del Módulo 200. Estas incluyen identificadores para la fusión, variables demográficas clave (`P203` para parentesco, `P207` para sexo, `P208A` para edad, `P209` para estado civil) y el factor de expansión de población (`FACPOB07`). La variable `P210` (condición de actividad) está comentada, lo que implica que no será incluida en el análisis.

### Selección de variables de interés

In [13]:
variables_modulo_200 = [
    # --- VARIABLES DE IDENTIFICACIÓN Y CONTEXTO ---
    "AÑO",         # Año de la encuesta (Control temporal)
    "MES",         # Mes de ejecución de la encuesta (Control estacional)
    "CONGLOME",    # Número de conglomerado (Llave de combinación con Módulo 100)
    "VIVIENDA",    # Número de selección de la vivienda (Llave de combinación con Módulo 100)
    "HOGAR",       # Número secuencial del hogar (Llave de combinación con Módulo 100)
    "CODPERSO",    # Número de orden de la persona (Identificador único intra-hogar)
    "UBIGEO",      # Ubicación geográfica (Establece distritos y regiones)
    "DOMINIO",     # Dominio Geográfico (Agrupación regional macros: Costa, Sierra, Selva)
    "ESTRATO",     # Estrato Geográfico / Tamaño poblacional (Densidad urbana vs rural)

    # --- CARACTERÍSTICAS INDIVIDUALES CRÍTICAS ---
    "P203",        # Relación de parentesco con el jefe(a) del hogar
    "P204",        # ¿Es miembro del hogar?
    "P207",        # Sexo
    "P208A",       # ¿Qué edad tiene en años cumplidos?
    "P209",        # ¿Cuál es su estado civil o conyugal?

    # --- CONDICIÓN DE ACTIVIDAD (LÍNEA BASE TRABAJO) ---
    #"P210",        # La semana pasada... ¿Estuvo trabajando o realizando tareas para obtener ingresos?
    "FACPOB07"     # Factor de expansión anual de población
]

enaho_200 = enaho_200[variables_modulo_200]
print(f"✅ Selección completa. Forma del dataframe: {enaho_200.shape}")


✅ Selección completa. Forma del dataframe: (115145, 15)


#### Diagnóstico de Columnas del Módulo 200

Al igual que con el Módulo 100, se realiza un diagnóstico para clasificar las columnas del Módulo 200 en binarias, categóricas y numéricas, y para identificar las columnas administrativas. Esto ayuda a comprender la composición de los datos y a planificar las recodificaciones necesarias.

### Diagnóstico de columnas

#### Conversión de Variables Numéricas Continuas

En este paso, las variables `MES` y `P208A` (edad) se convierten a un tipo numérico flotante. Se utiliza `errors='coerce'` para manejar posibles valores no numéricos, convirtiéndolos en `NaN`.

In [14]:

total_columns_shape = enaho_200.shape[1]

binary_cols = []
categorical_cols = []
numeric_cols = []
skipped_cols = []

administrative_cols = ['ubigeo', 'factor', 'conglome', 'vivienda', 'hogar', 'codperso', 'estrato', 'dominio']

for col in enaho_200.columns:
    if col.lower() in administrative_cols:
        skipped_cols.append(col)
        continue

    unique_vals = enaho_200[col].dropna().unique()
    num_unique = len(unique_vals)

    if num_unique == 0:
        skipped_cols.append(col)
    elif num_unique == 1:
        skipped_cols.append(col)
    elif num_unique == 2:
        binary_cols.append(col)
    elif enaho_200[col].dtype in ['object', 'category'] or num_unique < 15:
        categorical_cols.append(col)
    else:
        numeric_cols.append(col)

total_accounted_for = len(binary_cols) + len(categorical_cols) + len(numeric_cols) + len(skipped_cols)
all_accounted_columns = binary_cols + categorical_cols + numeric_cols + skipped_cols
missing_columns = [col for col in enaho_200.columns if col not in all_accounted_columns]

print(f"Total Columns according to enaho_200.shape[1]: {total_columns_shape}")
print(f"Total Columns accounted for by loop:    {total_accounted_for}")
print("-" * 45)
print(f"🔗 Binary Columns:      {len(binary_cols)}")
print(f"📊 Categorical Columns: {len(categorical_cols)}")
print(f"📈 Numeric Columns:     {len(numeric_cols)}")
print(f"⚙️ Administrative/Empty: {len(skipped_cols)}")
print("-" * 45)

if len(missing_columns) > 0:
    print(f"⚠️ WARNING: There are {len(missing_columns)} columns completely missing from your lists!")
    print(f"The missing column names are: {missing_columns}")
else:
    print("✅ Success! The math balances perfectly. Every column is accounted for.")



Total Columns according to enaho_200.shape[1]: 15
Total Columns accounted for by loop:    15
---------------------------------------------
🔗 Binary Columns:      2
📊 Categorical Columns: 3
📈 Numeric Columns:     2
⚙️ Administrative/Empty: 8
---------------------------------------------
✅ Success! The math balances perfectly. Every column is accounted for.


### Conversión de valores en columnas numéricas

#### Aplicación de Conversión a Numérico

La variable `MES` se convierte a un formato numérico flotante. La `P208A` (edad) también se convierte para asegurar que se pueda realizar un análisis numérico con ella. Se verifica que la conversión se haya completado correctamente.

In [15]:

enaho_200_recoded = enaho_200.copy()

# MES: mes de ejecución de la encuesta, escala 1-12
enaho_200_recoded["MES"] = pd.to_numeric(enaho_200_recoded["MES"], errors="coerce").astype(float)

# P208A: edad en años cumplidos, escala continua
enaho_200_recoded["P208A"] = pd.to_numeric(enaho_200_recoded["P208A"], errors="coerce").astype(float)

print("✅ Conversión de variables numéricas continuas completada.")


✅ Conversión de variables numéricas continuas completada.


#### Conversión de Variables Binarias del Módulo 200

Las variables `P204` (miembro del hogar) y `P207` (sexo) se recodifican a un formato binario (1.0 para afirmativo, 0.0 para negativo). Específicamente, `P207` se mapea para que 'Mujer' sea 1.0, lo cual es relevante para el enfoque de género del proyecto. La variable `P210` (trabajó la semana pasada) está comentada, lo que indica que no se está recodificando.

### Conversión de valores en columnas binarias

#### Aplicación de Recodificaciones Binarias

Los valores de `P204` se mapean a 1.0 para 'Si' y 0.0 para 'No'. Los valores de `P207` se mapean a 1.0 para 'Mujer' y 0.0 para 'Hombre'. La recodificación de `P210` está comentada y no se aplica. Se confirma que estas recodificaciones se han completado.

In [16]:

# P204: Miembro del hogar (Si = 1.0, No = 0.0)
enaho_200_recoded["P204"] = enaho_200_recoded["P204"].map({"Si": 1.0, "No": 0.0}).astype(float)

# P207: Sexo (Mujer = 1.0, Hombre = 0.0) -> orientado al proyecto de violencia de género
enaho_200_recoded["P207"] = enaho_200_recoded["P207"].map({"Mujer": 1.0, "Hombre": 0.0}).astype(float)

# P210: ¿Trabajó la semana pasada? (Si = 1.0, No = 0.0)
# enaho_200_recoded["P210"] = enaho_200_recoded["P210"].map({"Si": 1.0, "No": 0.0}).astype(float)

print("✅ Recodificación binaria de P204 y P207 completada.")



✅ Recodificación binaria de P204 y P207 completada.


#### Conversión de Variables Categóricas del Módulo 200

Las variables `P203` (parentesco con el jefe del hogar) y `P209` (estado civil) se recodifican para generar indicadores binarios. `P203` se convierte a 1.0 si la persona es 'Jefe/Jefa' y 0.0 de lo contrario. `P209` se convierte a 1.0 si la persona está 'Casado(a)' o 'Conviviente', y 0.0 de lo contrario, creando un indicador de 'en_pareja'.

### Conversión de valores en columnas categóricas

#### Aplicación de Recodificaciones Categóricas

Los valores de `P203` se transforman en una variable binaria (`es_jefe`). Los valores de `P209` se transforman en una variable binaria (`en_pareja`). Se confirma la finalización de estas recodificaciones y la estructura actual del DataFrame `enaho_200_recoded`.

In [17]:
enaho_200_recoded["P203"] = enaho_200_recoded["P203"].isin(["Jefe/Jefa"]).astype(float)

# P209 (Estado civil): Casado(a) o Conviviente pasa a 1.0, el resto a 0.0 (luego "en_pareja")
enaho_200_recoded["P209"] = enaho_200_recoded["P209"].isin(["Casado(a)", "Conviviente"]).astype(float)

print("✅ Recodificación de P203 y P209 completada.")
print(f"Estructura actual (nombres originales intactos): {enaho_200_recoded.shape}")



✅ Recodificación de P203 y P209 completada.
Estructura actual (nombres originales intactos): (115145, 15)


#### Renombramiento de Columnas del Módulo 200

Se renombran las columnas del Módulo 200 a nombres más cortos y descriptivos para facilitar el análisis. Esto incluye los identificadores (`anio`, `mes`, `ubigeo`, `cod_persona`) y las variables recodificadas (`es_jefe`, `es_mujer`, `edad`, `en_pareja`, `trabajo_ingresos`, `factor_poblacion`). El resultado es el DataFrame `enaho_200_final`.

### Renombrar columnas

#### Aplicación del Renombramiento de Columnas

Las columnas del DataFrame `enaho_200_recoded` se renombran utilizando el `diccionario_renombrar_200`. Esto asegura que las variables tengan nombres consistentes y legibles para las siguientes etapas del pipeline. Se muestra la lista de las nuevas columnas para verificación.

In [18]:

diccionario_renombrar_200 = {
    # --- VARIABLES DE IDENTIFICACIÓN Y CONTEXTO ---
    "AÑO": "anio",
    "MES": "mes",
    "CONGLOME": "conglome",
    "VIVIENDA": "vivienda",
    "HOGAR": "hogar",
    "CODPERSO": "cod_persona",
    "UBIGEO": "ubigeo",
    "DOMINIO": "dominio",
    "ESTRATO": "estrato",

    # --- CARACTERÍSTICAS INDIVIDUALES MAPEADAS ---
    "P203": "es_jefe",               # 1.0 si es Jefe/Jefa de hogar
    "P204": "ind_miembro_hogar",     # 1.0 si es residente habitual
    "P207": "es_mujer",              # 1.0 si es mujer
    "P208A": "edad",                 # Escala continua en años
    "P209": "en_pareja",             # 1.0 si es casado(a) o conviviente

    # --- CONDICIÓN DE ACTIVIDAD Y EXPANSIÓN ---
    "P210": "trabajo_ingresos",      # 1.0 si trabajó la semana pasada
    "FACPOB07": "factor_poblacion"   # Factor de expansión poblacional (individuos)
}

enaho_200_final = enaho_200_recoded.rename(columns=diccionario_renombrar_200)

print("✅ Módulo 200: columnas renombradas exitosamente.")
print("Nuevas columnas en el dataset 200:\n", list(enaho_200_final.columns))


✅ Módulo 200: columnas renombradas exitosamente.
Nuevas columnas en el dataset 200:
 ['anio', 'mes', 'conglome', 'vivienda', 'hogar', 'cod_persona', 'ubigeo', 'dominio', 'estrato', 'es_jefe', 'ind_miembro_hogar', 'es_mujer', 'edad', 'en_pareja', 'factor_poblacion']


### Módulo 300: Educación

El Módulo 300 de la ENAHO contiene información sobre el nivel educativo, alfabetización y acceso a tecnologías de la información a nivel individual. Esta sección carga, selecciona y prepara estas variables.

## Módulo 300: educación

#### Carga de Datos del Módulo 300

Cargamos el archivo `.sav` correspondiente al Módulo 300 (`Enaho01A-2025-300.sav`).

In [19]:
enaho_300 = pd.read_spss('../data/raw/Enaho01A-2025-300.sav')

#### Selección de Variables del Módulo 300

Se seleccionan las variables de interés del Módulo 300, que incluyen identificadores de fusión, logro educativo (`P301A`), alfabetización (`P302`), lengua materna (`P300A`), uso de internet (`P316$1`), posesión de celular (`P316A1`) y el factor de expansión de educación (`FACTORA07`).

### Selección de variables de interés

#### Diagnóstico de Columnas del Módulo 300

Se realiza un diagnóstico de los tipos de variables en el Módulo 300 para clasificar las columnas y entender su naturaleza, lo cual es esencial para las recodificaciones posteriores.

In [20]:
variables_modulo_300 = [
    # --- LLAVES INDISPENSABLES DE FUSIÓN (Fusión con Módulos 100 y 200) ---
    "AÑO",          # Año de la encuesta (Control temporal)
    "MES",          # Mes de ejecución de la encuesta
    "CONGLOME",     # Número de conglomerado
    "VIVIENDA",     # Número de selección de la vivienda
    "HOGAR",        # Número secuencial del hogar
    "DOMINIO",      # Dominio Geográfico (Costa, Sierra, Selva, etc.)
    "ESTRATO",      # Estrato Geográfico / Tamaño de la población
    "CODPERSO",     # Número de orden de la persona (Match exacto por individuo)
    "UBIGEO",       # Ubicación geográfica (Distrito)

    # --- LOGRO EDUCATIVO Y CAPITAL HUMANO (Línea Base del Individuo) ---
    "P301A",        # Último año, grado de estudios o nivel educativo aprobado (Categoría clave)

    # --- ALFABETIZACIÓN FUNCIONAL Y LINGÜÍSTICA ---
    "P302",          # ¿Sabe leer y escribir? (Alfabetización funcional)
    "P300A",          # Lengua materna

    # --- TECNOLOGÍAS DE LA INFORMACIÓN Y ALFABETIZACIÓN DIGITAL ---
    "P316$1",       # Uso de Internet en los últimos 3 meses (Conectividad individual)
    "P316A1",       # Uso de teléfono celular propio en el mes anterior (Autonomía de comunicación)

    # --- FACTOR DE PONDERACIÓN INDIVIDUAL ---
    "FACTORA07"     # Factor de expansión anual de población del Módulo de Educación
]

enaho_300 = enaho_300[variables_modulo_300]

#### Aplicación del Diagnóstico de Columnas

El código de diagnóstico clasifica las columnas en binarias, categóricas, numéricas y administrativas/vacías, proporcionando un resumen de la distribución de tipos de datos en `enaho_300`. Se confirma que todas las columnas han sido contabilizadas.

### Diagnóstico inicial de columnas

In [21]:

total_columns_shape = enaho_300.shape[1]

binary_cols = []
categorical_cols = []
numeric_cols = []
skipped_cols = []

administrative_cols = ['ubigeo', 'factor', 'conglome', 'vivienda', 'hogar', 'codperso', 'estrato', 'dominio']

for col in enaho_300.columns:
    if col.lower() in administrative_cols:
        skipped_cols.append(col)
        continue

    unique_vals = enaho_300[col].dropna().unique()
    num_unique = len(unique_vals)

    if num_unique == 0:
        skipped_cols.append(col)
    elif num_unique == 1:
        skipped_cols.append(col)
    elif num_unique == 2:
        binary_cols.append(col)
    elif enaho_300[col].dtype in ['object', 'category'] or num_unique < 15:
        categorical_cols.append(col)
    else:
        numeric_cols.append(col)

total_accounted_for = len(binary_cols) + len(categorical_cols) + len(numeric_cols) + len(skipped_cols)
all_accounted_columns = binary_cols + categorical_cols + numeric_cols + skipped_cols
missing_columns = [col for col in enaho_300.columns if col not in all_accounted_columns]

print(f"Total Columns according to enaho_300.shape[1]: {total_columns_shape}")
print(f"Total Columns accounted for by loop:    {total_accounted_for}")
print("-" * 45)
print(f"🔗 Binary Columns:      {len(binary_cols)}")
print(f"📊 Categorical Columns: {len(categorical_cols)}")
print(f"📈 Numeric Columns:     {len(numeric_cols)}")
print(f"⚙️ Administrative/Empty: {len(skipped_cols)}")
print("-" * 45)

if len(missing_columns) > 0:
    print(f"⚠️ WARNING: There are {len(missing_columns)} columns completely missing from your lists!")
    print(f"The missing column names are: {missing_columns}")
else:
    print("✅ Success! The math balances perfectly. Every column is accounted for.")



Total Columns according to enaho_300.shape[1]: 15
Total Columns accounted for by loop:    15
---------------------------------------------
🔗 Binary Columns:      3
📊 Categorical Columns: 3
📈 Numeric Columns:     1
⚙️ Administrative/Empty: 8
---------------------------------------------
✅ Success! The math balances perfectly. Every column is accounted for.


#### Conversión de Variables Numéricas Continuas

La variable `MES` del Módulo 300 se convierte a un tipo numérico flotante para asegurar la consistencia en el tratamiento de los meses a través de los diferentes módulos.

### Conversión de valores en columnas numéricas

#### Aplicación de Conversión a Numérico

Se convierte la columna `MES` a formato numérico flotante. Se confirma la finalización de esta conversión.

In [22]:

enaho_300_recoded = enaho_300.copy()

# MES: mes de ejecución de la encuesta, escala 1-12
enaho_300_recoded["MES"] = pd.to_numeric(enaho_300_recoded["MES"], errors="coerce").astype(float)

print("✅ Conversión de MES a numérico continuo completada.")



✅ Conversión de MES a numérico continuo completada.


#### Manejo de Valores Faltantes en `P302` y Recodificación Binaria

Se rellena los valores `NaN` en la columna `P302` (Sabe leer y escribir) con 'Si', asumiendo que los valores faltantes implican una respuesta afirmativa o por una consideración específica del dataset. Luego, se realiza una recodificación binaria de `P316$1` (uso de Internet), `P316A1` (uso de teléfono celular propio) y la misma `P302` (sabe leer y escribir) a 0s y 1s.

### Conversión de valores en columnas binarias

#### Recodificación Binaria de `P302`

Se muestran los conteos de `P302` antes de la recodificación binaria, incluyendo los valores `NaN` que han sido rellenados. Esto valida el paso de imputación de valores faltantes.

In [23]:
enaho_300_recoded['P302'] = enaho_300_recoded['P302'].fillna('Si')

enaho_300_recoded[['P302']].value_counts(dropna=False)

P302
Si      91558
No      12888
Name: count, dtype: int64

#### Aplicación de Recodificaciones Binarias del Módulo 300

Las variables `P316$1`, `P316A1` y `P302` se recodifican a 1.0 (afirmativo) y 0.0 (negativo) utilizando el método `.map()`. Se confirma la finalización de las recodificaciones binarias.

In [24]:

# P316$1: Habilidad / uso digital (Si = 1.0, No = 0.0)
enaho_300_recoded["P316$1"] = (
    enaho_300_recoded["P316$1"].map({"Si": 1.0, "No": 0.0}).astype(float)
)

# P316A1: Acceso a Internet vía celular propio (Telf.celular propio. = 1.0, Pase = 0.0)
enaho_300_recoded["P316A1"] = (
    enaho_300_recoded["P316A1"]
    .map({"Telf.celular propio.": 1.0, "Pase": 0.0})
    .astype(float)
)

enaho_300_recoded['P302'] = enaho_300_recoded['P302'].map({'Si': 1.0, 'No': 0.0}).astype(float)

print("✅ Recodificación binaria de P316$1 y P316A1 completada.")



✅ Recodificación binaria de P316$1 y P316A1 completada.


#### Conversión de Variables Categóricas del Módulo 300

Esta sección se enfoca en crear indicadores binarios a partir de variables categóricas. Se generan dos nuevas variables a partir de `P301A` (Nivel educativo): `edu_superior` (1.0 si tiene educación superior o posgrado) y `edu_basica` (1.0 si su máximo nivel fue educación básica). Además, se crea `P300A_nativa` (1.0 si la lengua materna es una lengua nativa peruana) a partir de `P300A`.

### Conversión de valores en columnas categóricas

#### Aplicación de Recodificaciones Categóricas del Módulo 300

Se calculan las variables binarias `edu_superior`, `edu_basica` y `lengua_materna_nativa` basándose en los criterios definidos. Se confirma que estas variables se han calculado correctamente y se muestra la estructura actualizada de `enaho_300_recoded`.

In [25]:

educacion_raw = enaho_300_recoded["P301A"]

# Nivel superior: tecnológico, universitario o posgrado (completo o incompleto)
lista_superior = [
    "Superior no univ. completa",
    "Superior univ. completa",
    "Superior univ. incompleta",
    "Superior no univ. Incompleta",
    "Maestria / Doctorado"
]
enaho_300_recoded["P301A"] = educacion_raw.isin(lista_superior).astype(float)

# Nivel básico: el máximo nivel alcanzado quedó dentro del marco escolar básico
lista_basica = [
    "Educacion inicial",
    "Primaria incompleta",
    "Primaria completa",
    "Secundaria incompleta",
    "Secundaria completa",
    "Basica especial"
]
enaho_300_recoded["P301A_basica"] = educacion_raw.isin(lista_basica).astype(float)


lista_nativas = [
    "Quechua",
    "Aimara",
    "Ashaninka",
    "Awajun/Aguarun",
    "Shipibo – Konibo",
    "Shawi/Chayahuita",
    "Matsigenka/Machiguenga",
    "Achuar",
    "Otra lengua nativa",
]
enaho_300_recoded["P300A_nativa"] = enaho_300_recoded["P300A"].isin(lista_nativas).astype(float)



print("✅ Variables de nivel educativo (superior / básica) calculadas correctamente.")
print(f"Estructura actual del DataFrame 300: {enaho_300_recoded.shape}")



✅ Variables de nivel educativo (superior / básica) calculadas correctamente.
Estructura actual del DataFrame 300: (104446, 17)


#### Renombramiento de Columnas del Módulo 300

Se renombran las columnas del Módulo 300 para estandarizar los nombres de las variables, siguiendo una convención de minúsculas y sufijos descriptivos. Se elimina la columna original `lengua_materna` ya que se ha creado una versión binaria (`lengua_materna_nativa`). El resultado es el DataFrame `enaho_300_final`.

### Renombrar variables

#### Aplicación del Renombramiento de Columnas y Eliminación de Duplicados

Las columnas del DataFrame `enaho_300_recoded` se renombran utilizando el `diccionario_renombrar_300`. Posteriormente, la columna `lengua_materna` se elimina, ya que la información clave ha sido capturada en la nueva variable `lengua_materna_nativa`. Se muestra la lista de las nuevas columnas para verificación.

In [26]:
diccionario_renombrar_300 = {
    # --- LLAVES INDISPENSABLES DE FUSIÓN ---
    "AÑO": "anio",
    "MES": "mes",
    "CONGLOME": "conglome",
    "VIVIENDA": "vivienda",
    "HOGAR": "hogar",
    "DOMINIO": "dominio",
    "ESTRATO": "estrato",
    "CODPERSO": "cod_persona",
    "UBIGEO": "ubigeo",

    # --- LOGRO EDUCATIVO Y CAPITAL HUMANO ---
    "P301A": "edu_superior",         # 1.0 si alcanzó nivel superior o posgrado
    "P301A_basica": "edu_basica",    # 1.0 si su máximo nivel fue el marco escolar básico

    # --- ALFABETIZACIÓN FUNCIONAL Y LINGÜÍSTICA ---
    "P302": "alfabetizacion_funcional",  # 1.0 si sabe leer y escribir (alfabetización funcional)
    "P300A": "lengua_materna",           # Lengua materna (categoría original, se puede recodificar a nativa vs no nativa)
    "P300A_nativa": "lengua_materna_nativa",  # 1.0 si la lengua materna es una lengua nativa peruana

    # --- TECNOLOGÍAS DE LA INFORMACIÓN Y ALFABETIZACIÓN DIGITAL ---
    "P316$1": "tic_uso_internet",    # 1.0 si usó internet en los últimos 3 meses
    "P316A1": "tic_celular_propio",  # 1.0 si usó celular propio el mes anterior

    # --- FACTOR DE PONDERACIÓN INDIVIDUAL ---
    "FACTORA07": "factor_educacion"  # Factor de expansión de este módulo
}

enaho_300_final = enaho_300_recoded.rename(columns=diccionario_renombrar_300)
enaho_300_final.drop(columns=["lengua_materna"], inplace=True)  # Eliminamos la columna original de lengua materna, ya que ahora tenemos la versión nativa vs no nativa

print("✅ Módulo 300: columnas renombradas exitosamente.")
print("Nuevas columnas en el dataset 300:\n", list(enaho_300_final.columns))


✅ Módulo 300: columnas renombradas exitosamente.
Nuevas columnas en el dataset 300:
 ['anio', 'mes', 'conglome', 'vivienda', 'hogar', 'dominio', 'estrato', 'cod_persona', 'ubigeo', 'edu_superior', 'alfabetizacion_funcional', 'tic_uso_internet', 'tic_celular_propio', 'factor_educacion', 'edu_basica', 'lengua_materna_nativa']


### Módulo 400: Salud

El Módulo 400 de la ENAHO recopila información sobre el estado de salud de los individuos, incluyendo enfermedades crónicas, discapacidades, y acceso a seguros de salud. Esta sección carga, selecciona y procesa estas variables.

## Módulo 400: salud

#### Carga de Datos del Módulo 400

Cargamos el archivo `.sav` correspondiente al Módulo 400 (`Enaho01A-2025-400.sav`).

In [27]:
enaho_400 = pd.read_spss('../data/raw/Enaho01A-2025-400.sav')

#### Visualización de Columnas del Módulo 400

Se imprimen todas las columnas del DataFrame `enaho_400` para una inspección rápida y para asegurar que los nombres de las variables a seleccionar sean correctos.

In [28]:
print(list(enaho_400.columns))

['AÑO', 'MES', 'CONGLOME', 'VIVIENDA', 'HOGAR', 'CODPERSO', 'UBIGEO', 'DOMINIO', 'ESTRATO', 'CODINFOR', 'P400N', 'P400I', 'P400A1', 'P400A2', 'P400A3', 'P401C', 'P401D1', 'P401D2', 'P401D3', 'P401D4', 'P401D5', 'P401D6', 'P401D7', 'P401D8', 'P401D9', 'P401F', 'P401G', 'P401G1', 'P401G2', 'P401H1', 'P401H2', 'P401H3', 'P401H4', 'P401H5', 'P401H6', 'P401', 'P4021', 'P4022', 'P4023', 'P4024', 'P4025', 'P4026', 'P4031', 'P4032', 'P4033', 'P4034', 'P4035', 'P4036', 'P4037', 'P4038', 'P4039', 'P40310', 'P40311', 'P40313', 'P40314', 'P4041', 'P4042', 'P4043', 'P4044', 'P4045', 'P4046', 'P4047', 'P4091', 'P4092', 'P4093', 'P4094', 'P4095', 'P4096', 'P4097', 'P4098', 'P4099', 'P40910', 'P40911', 'P413B1', 'P413B1A', 'P413B2', 'P413B2A', 'P413D1', 'P413D1A', 'P413D2', 'P413D2A', 'P414N$01', 'P414N$02', 'P414N$03', 'P414N$04', 'P414N$05', 'P414N$06', 'P414N$07', 'P414N$08', 'P414N$09', 'P414N$10', 'P414N$11', 'P414N$12', 'P414N$13', 'P414N$14', 'P414N$15', 'P414N$16', 'P414$01', 'P414$02', 'P414$

#### Selección de Variables del Módulo 400

Se seleccionan las variables relevantes del Módulo 400, que incluyen identificadores, controles sociodemográficos (que se encuentran duplicados con Módulo 200, pero se manejarán en la fusión), la variable de enfermedad crónica (`P401`), un conjunto de variables de discapacidad (`P401H1` a `P401H6`), variables de seguros de salud (`P4191` a `P4198`) y el factor de expansión (`FACTOR07`).

### Selección de variables de interés

#### Diagnóstico de Columnas del Módulo 400

Se realiza un diagnóstico de los tipos de variables en el Módulo 400 para clasificar las columnas y entender su naturaleza, lo cual es esencial para las recodificaciones posteriores.

In [29]:
variables_modulo_400 = [
    # --- LLAVES INDISPENSABLES ---
    "AÑO", "MES", "CONGLOME", "VIVIENDA", "HOGAR", "CODPERSO", "UBIGEO",

    # --- CONTROLES SOCIODEMOGRÁFICOS PRESENTES ---
    "DOMINIO", "ESTRATO", "P203", "P207", "P208A", "P209",

    # --- ENFERMEDAD CRÓNICA ---
    "P401",  # ¿Sufre de alguna enfermedad o malestar crónico?

    # --- DISCAPACIDAD / LIMITACIONES PERMANENTES (Serie P401H) ---
    "P401H1",  # Moverse o caminar / usar brazos o piernas
    "P401H2",  # Ver, aun usando anteojos
    "P401H3",  # Hablar o comunicarse
    "P401H4",  # Oír, aún usando audífonos
    "P401H5",  # Entender o aprender (concentrarse y recordar)
    "P401H6",  # Relacionarse con los demás (emociones/conductas)

    # --- SEGUROS DE SALUD ---
    "P4191",  # ¿Tiene ESSALUD?
    "P4192",  # ¿Tiene seguro privado de salud?
    "P4193",  # ¿Tiene seguro de EPS?
    "P4194",  # ¿Tiene Seguro de FFAA?
    "P4195",  # ¿Tiene Seguro Integral de Salud?
    "P4196",  # ¿Tiene seguro universitario?
    "P4197",  # ¿Tiene seguro escolar privado?
    "P4198",  # ¿Tiene algún otro tipo de seguro?

    # --- FACTOR DE EXPANSION DISPONIBLE EN ESTE ARCHIVO ---
    "FACTOR07"
]



enaho_400 = enaho_400[variables_modulo_400]

#### Aplicación del Diagnóstico de Columnas

El código de diagnóstico clasifica las columnas en binarias, categóricas, numéricas y administrativas/vacías, proporcionando un resumen de la distribución de tipos de datos en `enaho_400`. Se confirma que todas las columnas han sido contabilizadas.

### Diagnóstico inicial de columnas

In [30]:

total_columns_shape = enaho_400.shape[1]

binary_cols = []
categorical_cols = []
numeric_cols = []
skipped_cols = []

administrative_cols = ['ubigeo', 'factor', 'conglome', 'vivienda', 'hogar', 'codperso', 'estrato', 'dominio']

for col in enaho_400.columns:
    if col.lower() in administrative_cols:
        skipped_cols.append(col)
        continue

    unique_vals = enaho_400[col].dropna().unique()
    num_unique = len(unique_vals)

    if num_unique == 0:
        skipped_cols.append(col)
    elif num_unique == 1:
        skipped_cols.append(col)
    elif num_unique == 2:
        binary_cols.append(col)
    elif enaho_400[col].dtype in ['object', 'category'] or num_unique < 15:
        categorical_cols.append(col)
    else:
        numeric_cols.append(col)

total_accounted_for = len(binary_cols) + len(categorical_cols) + len(numeric_cols) + len(skipped_cols)
all_accounted_columns = binary_cols + categorical_cols + numeric_cols + skipped_cols
missing_columns = [col for col in enaho_400.columns if col not in all_accounted_columns]

print(f"Total Columns according to enaho_400.shape[1]: {total_columns_shape}")
print(f"Total Columns accounted for by loop:    {total_accounted_for}")
print("-" * 45)
print(f"🔗 Binary Columns:      {len(binary_cols)}")
print(f"📊 Categorical Columns: {len(categorical_cols)}")
print(f"📈 Numeric Columns:     {len(numeric_cols)}")
print(f"⚙️ Administrative/Empty: {len(skipped_cols)}")
print("-" * 45)

if len(missing_columns) > 0:
    print(f"⚠️ WARNING: There are {len(missing_columns)} columns completely missing from your lists!")
    print(f"The missing column names are: {missing_columns}")
else:
    print("✅ Success! The math balances perfectly. Every column is accounted for.")



Total Columns according to enaho_400.shape[1]: 29
Total Columns accounted for by loop:    29
---------------------------------------------
🔗 Binary Columns:      16
📊 Categorical Columns: 3
📈 Numeric Columns:     2
⚙️ Administrative/Empty: 8
---------------------------------------------
✅ Success! The math balances perfectly. Every column is accounted for.


#### Conversión de Variables Binarias del Módulo 400

Esta sección recodifica las variables binarias del Módulo 400 a 0s y 1s. Incluye `P207` (sexo, mapeando 'Mujer' a 1.0), `P401` y las variables `P401H` (síntomas/discapacidad), y las variables `P419` (cobertura de seguros). Se utiliza el método `.map()` para aplicar las recodificaciones.

### Conversión de valores en columnas binarias

#### Aplicación de Recodificaciones Binarias del Módulo 400

Las variables `P207`, las de síntomas/discapacidad (`P401` y `P401H1` a `P401H6`) y las de seguros (`P4191` a `P4198`) se recodifican a formato binario. Se confirma que estas recodificaciones se han completado correctamente.

In [31]:
enaho_400_recoded = enaho_400.copy()

# P207: Sexo (Mujer = 1.0, Hombre = 0.0)
enaho_400_recoded["P207"] = enaho_400_recoded["P207"].map({"Mujer": 1.0, "Hombre": 0.0}).astype(float)

# P401 y desagregaciones P401H (Si = 1.0, No = 0.0)
columnas_sintomas = ["P401", "P401H1", "P401H2", "P401H3", "P401H4", "P401H5", "P401H6"]
for col in columnas_sintomas:
    if col in enaho_400_recoded.columns:
        enaho_400_recoded[col] = enaho_400_recoded[col].map({"Si": 1.0, "No": 0.0}).astype(float)

# Cobertura de seguros (P419): 1.0 = tiene el seguro específico, 0.0 = No
mapeos_seguros = {
    "P4191": {"EsSalud": 1.0, "No": 0.0},
    "P4192": {"Seguro Privado de Salud": 1.0, "No": 0.0},
    "P4193": {"Entidad Prestadora de Salud": 1.0, "No": 0.0},
    "P4194": {"Seguro FF.AA./PNP": 1.0, "No": 0.0},
    "P4195": {"Seguro Integral de Salud (SIS)": 1.0, "No": 0.0},
    "P4196": {"Seguro Universitario": 1.0, "No": 0.0},
    "P4197": {"Seguro Escolar Privado": 1.0, "No": 0.0},
    "P4198": {"Otro": 1.0, "No": 0.0}
}

for col, mapa in mapeos_seguros.items():
    if col in enaho_400_recoded.columns:
        enaho_400_recoded[col] = enaho_400_recoded[col].map(mapa).astype(float)

print("✅ Recodificación binaria de sexo, síntomas/discapacidad y seguros completada.")



✅ Recodificación binaria de sexo, síntomas/discapacidad y seguros completada.


#### Conversión de Variables Categóricas del Módulo 400

Las variables `MES`, `P203` (parentesco) y `P209` (estado civil) se recodifican. `MES` se convierte a numérico. `P203` se convierte a 1.0 si es 'Jefe/Jefa' y 0.0 de lo contrario. `P209` se convierte a 1.0 si está 'Casado(a)' o 'Conviviente' y 0.0 de lo contrario.

### Conversión de valores en columnas categóricas

#### Aplicación de Recodificaciones Categóricas

Se convierten las columnas `MES`, `P203`, y `P209` a sus respectivos formatos numéricos o binarios. Se confirma la finalización de estas recodificaciones y la estructura del DataFrame `enaho_400_recoded`.

In [32]:

# MES: mes de ejecución de la encuesta, escala 1-12
enaho_400_recoded["MES"] = pd.to_numeric(enaho_400_recoded["MES"], errors="coerce").astype(float)

# P203 (Parentesco): Jefe/Jefa pasa a 1.0, el resto a 0.0
enaho_400_recoded["P203"] = enaho_400_recoded["P203"].isin(["Jefe/Jefa"]).astype(float)

# P209 (Estado civil): Casado(a) o Conviviente pasa a 1.0, el resto a 0.0
enaho_400_recoded["P209"] = enaho_400_recoded["P209"].isin(["Casado(a)", "Conviviente"]).astype(float)

print("✅ Recodificación de MES, P203 y P209 completada.")
print(f"Estructura actual del DataFrame 400: {enaho_400_recoded.shape}")



✅ Recodificación de MES, P203 y P209 completada.
Estructura actual del DataFrame 400: (107802, 29)


#### Creación de Variables Agregadas y Eliminación de Originales

Antes de renombrar, se crean dos nuevas variables binarias agregadas:

*   `P419`: Indica si la persona tiene **algún tipo de seguro de salud** (1.0 si tiene al menos uno de los seguros `P4191` a `P4198`). Las columnas de seguros individuales se eliminan.
*   `P401H`: Indica si la persona tiene **alguna limitación/discapacidad** (1.0 si tiene al menos una de las limitaciones `P401H1` a `P401H6`). Las columnas de limitaciones individuales se eliminan.

### Renombrar columnas

In [33]:
# 1. Define la lista de columnas binarias que quieres evaluar
columnas_seguro = ['P4191', 'P4192', 'P4193', 'P4194', 'P4195', 'P4196', 'P4197', 'P4198']

# 2. Crea la nueva columna condicional
enaho_400_recoded['P419'] = enaho_400_recoded[columnas_seguro].any(axis=1).astype(int)
enaho_400_recoded.drop(columns=columnas_seguro, inplace=True)  # Elimina las columnas individuales de seguros si ya no las necesitas

columnas_limitacion = ['P401H1', 'P401H2', 'P401H3', 'P401H4', 'P401H5', 'P401H6']

enaho_400_recoded['P401H'] = enaho_400_recoded[columnas_limitacion].any(axis=1).astype(int)
enaho_400_recoded.drop(columns=columnas_limitacion, inplace=True)  # Elimina las columnas individuales de limitaciones si ya no las necesitas

#### Renombramiento de Columnas del Módulo 400

Finalmente, se renombran las columnas del Módulo 400 a nombres más descriptivos y estandarizados, incluyendo los identificadores, las variables sociodemográficas recodificadas y las nuevas variables de salud agregadas (`salud_enfermedad_cronica`, `tiene_limitacion`, `tiene_seguro`). El factor de expansión se renombra a `factor_salud`. El resultado es el DataFrame `enaho_400_final`.

In [34]:

diccionario_renombrar_400 = {
    # --- LLAVES INDISPENSABLES ---
    "AÑO": "anio",
    "MES": "mes",
    "CONGLOME": "conglome",
    "VIVIENDA": "vivienda",
    "HOGAR": "hogar",
    "CODPERSO": "cod_persona",
    "UBIGEO": "ubigeo",

    # --- CONTROLES SOCIODEMOGRÁFICOS ---
    "DOMINIO": "dominio",
    "ESTRATO": "estrato",
    "P203": "es_jefe",               # 1.0 si es Jefe/Jefa de hogar
    "P207": "es_mujer",              # 1.0 si es mujer
    "P208A": "edad",                 # Escala continua en años
    "P209": "en_pareja",             # 1.0 si es casado(a) o conviviente

    # --- ENFERMEDAD CRÓNICA ---
    "P401": "salud_enfermedad_cronica",

    # --- DISCAPACIDAD / LIMITACIONES PERMANENTES (Serie P401H) ---
    "P401H": "tiene_limitacion",

    # --- SEGUROS DE SALUD ---
    "P419": "tiene_seguro",
    # --- FACTOR DE EXPANSIÓN ---
    "FACTOR07": "factor_salud"
}

enaho_400_final = enaho_400_recoded.rename(columns=diccionario_renombrar_400)

print("✅ Módulo 400: columnas renombradas exitosamente.")
print(f"Dimensiones finales del dataset 400: {enaho_400_final.shape}")
print("Columnas actuales:\n", list(enaho_400_final.columns))


✅ Módulo 400: columnas renombradas exitosamente.
Dimensiones finales del dataset 400: (107802, 17)
Columnas actuales:
 ['anio', 'mes', 'conglome', 'vivienda', 'hogar', 'cod_persona', 'ubigeo', 'dominio', 'estrato', 'es_jefe', 'es_mujer', 'edad', 'en_pareja', 'salud_enfermedad_cronica', 'factor_salud', 'tiene_seguro', 'tiene_limitacion']


### Agregación de Módulos a Nivel de Individuo

Esta sección se encarga de fusionar los módulos de ENAHO previamente procesados (`enaho_100_final`, `enaho_200_final`, `enaho_300_final`, `enaho_400_final`) en un único DataFrame a nivel de individuo. Esto implica combinar la información del hogar, de las características personales, educación y salud para cada persona encuestada. Se utilizan claves de combinación (`llaves_persona`, `llaves_individuo`) y se manejan columnas duplicadas para evitar conflictos.

## Agregación de columnas

#### Fusiones Iniciales de Módulos (200 + 300 + 400)

Primero, se fusionan los módulos `enaho_200_final` (miembros del hogar) y `enaho_300_final` (educación) usando `llaves_persona`. Luego, el resultado se fusiona con `enaho_400_final` (salud), excluyendo columnas que ya están presentes o se manejan de manera diferente para evitar sufijos `_x` y `_y` innecesarios. Estas fusiones se realizan con un `inner join` para mantener solo los registros que están presentes en todos los módulos.

### filtrado de solo mujeres

In [35]:
llaves_individuo = ["conglome", "vivienda", "hogar"]
llaves_persona   = ["conglome", "vivienda", "hogar", "cod_persona"]

df_personas = pd.merge(enaho_200_final, enaho_300_final, on=llaves_persona, how="inner")

df_personas = pd.merge(df_personas, enaho_400_final, on=llaves_persona, how="inner")

In [36]:
df_personas[['es_mujer_x', 'es_mujer_y']].value_counts(dropna=False)

es_mujer_x  es_mujer_y
1.0         1.0           54063
0.0         0.0           50383
Name: count, dtype: int64

In [37]:
df_personas.drop(columns=["anio_x", "mes_x", "ubigeo_x", "dominio_x", "estrato_x", "es_jefe_x", "es_mujer_x", "edad_x", "en_pareja_x", "anio_y", "mes_y", "ubigeo_y", "dominio_y", "estrato_y"], inplace=True)
df_personas.rename(columns={"es_jefe_y": "es_jefe",
                            "es_mujer_y": "es_mujer",
                            "edad_y": "edad",
                            "en_pareja_y": "en_pareja"}, inplace=True)

In [38]:
df_personas.shape

(104446, 26)

In [39]:
df_personas.columns

Index(['conglome', 'vivienda', 'hogar', 'cod_persona', 'ind_miembro_hogar',
       'factor_poblacion', 'edu_superior', 'alfabetizacion_funcional',
       'tic_uso_internet', 'tic_celular_propio', 'factor_educacion',
       'edu_basica', 'lengua_materna_nativa', 'anio', 'mes', 'ubigeo',
       'dominio', 'estrato', 'es_jefe', 'es_mujer', 'edad', 'en_pareja',
       'salud_enfermedad_cronica', 'factor_salud', 'tiene_seguro',
       'tiene_limitacion'],
      dtype='str')

In [40]:
mujeres_adultas = df_personas[(df_personas['es_mujer'] == 1.0) & (df_personas['edad'] >= 18)]

In [41]:
mujeres_adultas.shape

(40318, 26)

### Función agregadora

In [42]:
def calcular_media_distrital_ponderada(df, columna_valor, columna_factor, columna_ubigeo='ubigeo'):
    """
    Calcula la media ponderada de una variable continua, agrupada por distrito.
    """
    df_limpio = df.dropna(subset=[columna_valor, columna_factor, columna_ubigeo])

    numerador = (df_limpio[columna_valor] * df_limpio[columna_factor]).groupby(df_limpio[columna_ubigeo]).sum()
    denominador = df_limpio.groupby(columna_ubigeo)[columna_factor].sum()

    return (numerador / denominador.replace(0, np.nan)).rename('media')

In [43]:
import pandas as pd
import numpy as np

def calcular_proporcion_distrital_ponderada(df, columna_binaria, columna_factor, columna_ubigeo='ubigeo'):
    """
    Calcula la proporción poblacional ponderada agrupada por distrito (ubigeo).

    Parámetros:
    df (DataFrame): Dataset filtrado (ej. Mujeres > 18).
    columna_binaria (str): Columna con valores 0 y 1.
    columna_factor (str): Factor de expansión del módulo respectivo.
    columna_ubigeo (str): Nombre de la columna de código distrital.

    Retorna:
    Series: Proporción ponderada indexada por cada ubigeo.
    """
    columnas_requeridas = [columna_binaria, columna_factor, columna_ubigeo]

    # 1. Eliminar filas con nulos en las variables críticas
    df_limpio = df.dropna(subset=columnas_requeridas)

    # 2. Numerador y denominador ponderados, agrupados por distrito en un solo paso
    numerador = (df_limpio[columna_binaria] * df_limpio[columna_factor]).groupby(df_limpio[columna_ubigeo]).sum()
    denominador = df_limpio.groupby(columna_ubigeo)[columna_factor].sum()

    # 3. Proporción final, evitando división por cero
    proporcion = (numerador / denominador.replace(0, np.nan)).rename('proporcion')

    return proporcion

In [44]:
# 1. Mapeamos cada variable a su factor de expansión correspondiente
variables_factor_binarias = {
    # CARACTERÍSTICAS DE LOS MIEMBROS DEL HOGAR (Módulo 200)
    'es_jefe': 'factor_poblacion',
    #'ind_miembro_hogar': 'factor_poblacion',
    'es_mujer': 'factor_poblacion',
    #'edad': 'factor_poblacion',
    'en_pareja': 'factor_poblacion',
    #'trabajo_ingresos': 'factor_poblacion',


    # EDUCACIÓN (Módulo 300)
    'edu_superior': 'factor_educacion',
    'edu_basica': 'factor_educacion',
    'alfabetizacion_funcional': 'factor_educacion',
    'lengua_materna_nativa': 'factor_educacion',
    'tic_uso_internet': 'factor_educacion',
    'tic_celular_propio': 'factor_educacion',

    # SALUD (Módulo 400)
    'salud_enfermedad_cronica': 'factor_salud',

    'tiene_limitacion': 'factor_salud',
    'tiene_seguro': 'factor_salud',
    #'lim_discapacidad_motriz': 'factor_salud',
    #'lim_discapacidad_visual': 'factor_salud',
    #'lim_discapacidad_habla': 'factor_salud',
    #'lim_discapacidad_auditiva': 'factor_salud',
    #'lim_discapacidad_comprension': 'factor_salud',
    #'lim_discapacidad_relacion': 'factor_salud',
    #'seg_essalud': 'factor_salud',
    #'seg_privado': 'factor_salud',
    #'seg_eps': 'factor_salud',
    #'seg_ffaa_pnp': 'factor_salud',
    #'seg_sis': 'factor_salud',
    #'seg_universitario': 'factor_salud',
    #'seg_escolar_privado': 'factor_salud',
    #'seg_otro': 'factor_salud'

}

variables_factor_continuas = {
    'edad': 'factor_poblacion',
}



In [45]:
# 2. Calculamos todas las proporciones en un solo loop y las unimos con concat
df_distritos_mujeres = pd.concat(
    {
        f'prop_{col}': calcular_proporcion_distrital_ponderada(
            df=mujeres_adultas,
            columna_binaria=col,
            columna_factor=factor
        )
        for col, factor in variables_factor_binarias.items()
    },
    axis=1
)

# 3. Calculamos las medias ponderadas para las variables continuas y las unimos
df_distritos_mujeres_continuas = pd.concat(
    {
        f'media_{col}': calcular_media_distrital_ponderada(
            df=mujeres_adultas,
            columna_valor=col,
            columna_factor=factor
        )
        for col, factor in variables_factor_continuas.items()
    },
    axis=1
)

# 4. Unimos ambos resultados en un solo DataFrame por distrito
df_distritos_mujeres = df_distritos_mujeres.join(df_distritos_mujeres_continuas, how='outer')
df_distritos_mujeres.index.name = 'ubigeo'

print(df_distritos_mujeres.head())

        prop_es_jefe  prop_es_mujer  prop_en_pareja  prop_edu_superior  \
ubigeo                                                                   
010101      0.334482            1.0        0.456272           0.426559   
010104      0.125000            1.0        0.625000           0.000000   
010106      0.285714            1.0        0.714286           0.000000   
010109      0.222794            1.0        0.602521           0.120571   
010110      0.393529            1.0        0.457313           0.369073   

        prop_edu_basica  prop_alfabetizacion_funcional  \
ubigeo                                                   
010101         0.522272                       0.937497   
010104         1.000000                       1.000000   
010106         1.000000                       1.000000   
010109         0.821234                       0.745002   
010110         0.464479                       0.828042   

        prop_lengua_materna_nativa  prop_tic_uso_internet  \
ubigeo       

In [46]:
df_distritos_mujeres.reset_index(inplace=True)

In [47]:
import os

output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok=True)
df_distritos_mujeres.to_csv(os.path.join(output_dir, 'proporciones_distritales_mujeres_adultas.csv'), index=True)

## enaho - hogares

In [48]:
variables_binarias_hogar = {'viv_tipo':'factor',
                  #'viv_paredes':'factor',
                  #'viv_pisos':'factor',
                  #'viv_techos':'factor',
                  'tic_celular':'factor',
                   'tic_internet':'factor',
                   #'agua_procedencia':'factor',
                   'agua_todos_dias':'factor',
                   'serv_electricidad':'factor',
                   'nbi_vivienda':'factor',
                   'nbi_hacinamiento':'factor',
                   'nbi_sshh':'factor',
                   'nbi_educacion':'factor',
                   'nbi_dependencia':'factor'}

#variables_continuas_hogar = {'viv_hab_total':'factor',
#                            'viv_hab_dormir':'factor'}

#### Tipos de Datos del DataFrame `enaho_100_final`

Se muestra un resumen de los tipos de datos (`dtypes`) de cada columna en el DataFrame `enaho_100_final`. Esto es útil para verificar que las recodificaciones anteriores hayan aplicado los tipos de datos correctos y que las columnas estén listas para la agregación numérica.

In [49]:
enaho_100_final.dtypes

anio                      str
mes                   float64
ubigeo                    str
conglome                  str
vivienda                  str
hogar                     str
dominio              category
estrato              category
factor                float64
viv_tipo              float64
tic_celular           float64
tic_internet          float64
agua_todos_dias       float64
serv_electricidad     float64
nbi_vivienda          float64
nbi_hacinamiento      float64
nbi_sshh              float64
nbi_educacion         float64
nbi_dependencia       float64
dtype: object

#### Cálculo de Proporciones Distritales para Variables Binarias del Hogar

Se utiliza la función `calcular_proporcion_distrital_ponderada` para calcular la proporción ponderada de cada variable binaria del hogar, agrupada por distrito. El resultado se concatena en el DataFrame `df_distritos_hogares`. Las variables continuas del hogar (`viv_hab_total`, `viv_hab_dormir`) están comentadas y no se procesan en este paso.

In [50]:
df_distritos_hogares = pd.concat(
    {
        f'prop_{col}': calcular_proporcion_distrital_ponderada(
            df=enaho_100_final,
            columna_binaria=col,
            columna_factor=factor
        )
        for col, factor in variables_binarias_hogar.items()
    },
    axis=1
)
"""
# 3. Calculamos las medias ponderadas para las variables continuas y las unimos
df_distritos_hogares_continuas = pd.concat(
    {
        f'media_{col}': calcular_media_distrital_ponderada(
            df=enaho_100_final,
            columna_valor=col,
            columna_factor=factor
        )
        for col, factor in variables_continuas_hogar.items()
    },
    axis=1
)
"""
# 4. Unimos ambos resultados en un solo DataFrame por distrito
#df_distritos_hogares = df_distritos_hogares.join(df_distritos_hogares_continuas, how='outer')
df_distritos_hogares.index.name = 'ubigeo'

df_distritos_hogares.reset_index(inplace=True)
print(df_distritos_hogares.head())

   ubigeo  prop_viv_tipo  prop_tic_celular  prop_tic_internet  \
0  010101       0.677290          0.942543           0.927752   
1  010104       0.666667          1.000000           1.000000   
2  010106       1.000000          0.875000           0.750000   
3  010109       0.737556          0.865025           0.865025   
4  010110       0.659159          0.972946           0.972946   

   prop_agua_todos_dias  prop_serv_electricidad  prop_nbi_vivienda  \
0              0.731953                0.962448           0.029381   
1              1.000000                1.000000           0.000000   
2              1.000000                0.750000           0.000000   
3              1.000000                0.932512           0.383417   
4              1.000000                0.837675           0.162325   

   prop_nbi_hacinamiento  prop_nbi_sshh  prop_nbi_educacion  \
0               0.014239       0.035045                 0.0   
1               0.000000       0.000000                 0.0   

#### Resumen Descriptivo de Proporciones Distritales del Hogar

Se genera un resumen descriptivo (`.describe()`) del DataFrame `df_distritos_hogares`. Esto proporciona estadísticas clave (media, desviación estándar, mínimos, máximos, cuartiles) para cada una de las proporciones agregadas a nivel distrital para las variables del hogar.

In [51]:
df_distritos_hogares.describe()

,prop_viv_tipo,prop_tic_celular,prop_tic_internet,prop_agua_todos_dias,prop_serv_electricidad,prop_nbi_vivienda,prop_nbi_hacinamiento,prop_nbi_sshh,prop_nbi_educacion,prop_nbi_dependencia
count,1271.000000,1271.000000,1271.000000,1233.000000,1271.000000,1271.000000,1271.000000,1271.000000,1271.000000,1271.000000
mean,0.742745,0.911555,0.868399,0.870839,0.914738,0.081162,0.044270,0.102379,0.003775,0.005642
std,0.157002,0.121480,0.160670,0.251623,0.180603,0.151911,0.079749,0.164642,0.020292,0.026458
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.652768,0.875000,0.833258,0.869279,0.895907,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.750000,0.954799,0.918571,1.000000,1.000000,0.000000,0.000000,0.021391,0.000000,0.000000
75%,0.857143,1.000000,1.000000,1.000000,1.000000,0.109882,0.063622,0.133578,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.750000,1.000000,0.285713,0.375000


#### Unión de Datasets Distritales

Se fusionan los DataFrames de indicadores distritales de mujeres adultas (`df_distritos_mujeres`) con los indicadores distritales del hogar (`df_distritos_hogares`) utilizando el `ubigeo` (código de distrito) como clave. Se realiza un `inner join` para asegurar que solo se incluyan los distritos presentes en ambos DataFrames.

### unir datasets

#### Visualización de los Primeros Registros del Dataset Unificado

Se muestran las primeras filas (`.head()`) del DataFrame `df_distritos_mujeres_hogares` resultante de la fusión. Esto permite verificar la estructura del nuevo DataFrame, asegurando que las columnas de ambos orígenes estén presentes y que la unión se haya realizado correctamente.

In [52]:
# unir dataset distrital de mujeres con características del hoghar
df_distritos_mujeres_hogares = df_distritos_hogares.merge(df_distritos_mujeres, on='ubigeo', how='inner')

#### Guardado del Dataset Final Unificado

El DataFrame `df_distritos_mujeres_hogares`, que contiene los indicadores agregados por distrito para mujeres y hogares, se guarda en un archivo CSV (`proporciones_distritales_mujeres_hogares.csv`) en el directorio `../data/processed/`. Este archivo consolida los resultados de los análisis distritales y está listo para usos posteriores.

In [53]:
df_distritos_mujeres_hogares.head()

,ubigeo,prop_viv_tipo,prop_tic_celular,prop_tic_internet,prop_agua_todos_dias,prop_serv_electricidad,prop_nbi_vivienda,prop_nbi_hacinamiento,prop_nbi_sshh,prop_nbi_educacion,...,prop_edu_superior,prop_edu_basica,prop_alfabetizacion_funcional,prop_lengua_materna_nativa,prop_tic_uso_internet,prop_tic_celular_propio,prop_salud_enfermedad_cronica,prop_tiene_limitacion,prop_tiene_seguro,media_edad
0,010101,0.677290,0.942543,0.927752,0.731953,0.962448,0.029381,0.014239,0.035045,0.0,...,0.426559,0.522272,0.937497,0.0,0.838800,0.941529,0.679256,0.112473,0.980843,47.066860
1,010104,0.666667,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,1.000000,1.000000,0.0,0.914349,0.736773,0.500000,0.000000,1.000000,47.250000
2,010106,1.000000,0.875000,0.750000,1.000000,0.750000,0.000000,0.000000,0.000000,0.0,...,0.000000,1.000000,1.000000,0.0,NaN,1.000000,0.714286,0.142857,1.000000,59.285714
3,010109,0.737556,0.865025,0.865025,1.000000,0.932512,0.383417,0.000000,0.076683,0.0,...,0.120571,0.821234,0.745002,0.0,0.813716,0.784441,0.579753,0.000000,1.000000,46.969642
4,010110,0.659159,0.972946,0.972946,1.000000,0.837675,0.162325,0.000000,0.000000,0.0,...,0.369073,0.464479,0.828042,0.0,0.866948,0.981205,0.830911,0.027976,1.000000,52.438633


### Secciones Anteriores (Eliminar/Ignorar)

Esta es una sección que contiene código de pasos previos o experimentales que el usuario puede querer eliminar o ignorar en la versión final del pipeline.

In [54]:
df_distritos_mujeres_hogares.to_csv('../data/processed/proporciones_distritales_mujeres_hogares.csv', index=True)

#### Descripción del DataFrame `df_distritos_mujeres`

Se proporciona un resumen descriptivo del DataFrame `df_distritos_mujeres`. Esto es probablemente un remanente de una etapa anterior donde se quería inspeccionar este DataFrame de forma individual, antes de la fusión con los datos del hogar. Si esta inspección ya no es necesaria, esta celda podría ser eliminada.

### anterior - eliminar

#### Definición de Variables Binarias del Hogar

Se definen las variables binarias del hogar que se desean agregar a nivel distrital. Estas incluyen el tipo de vivienda, acceso a tecnologías (celular, internet), acceso a agua y electricidad, y los indicadores de Necesidades Básicas Insatisfechas (NBI) relacionados con la vivienda. Cada variable se asocia con el factor de expansión `factor` del Módulo 100.

In [55]:
df_distritos_mujeres.describe()

,prop_es_jefe,prop_es_mujer,prop_en_pareja,prop_edu_superior,prop_edu_basica,prop_alfabetizacion_funcional,prop_lengua_materna_nativa,prop_tic_uso_internet,prop_tic_celular_propio,prop_salud_enfermedad_cronica,prop_tiene_limitacion,prop_tiene_seguro,media_edad
count,1271.000000,1271.0,1271.000000,1271.000000,1271.000000,1271.000000,1271.000000,1243.000000,1271.000000,1271.000000,1271.000000,1271.000000,1271.000000
mean,0.299761,1.0,0.576326,0.224900,0.673086,0.856309,0.333904,0.727288,0.814937,0.604005,0.081047,0.955024,48.926946
std,0.151753,0.0,0.165225,0.195033,0.185816,0.149277,0.374775,0.269794,0.172938,0.188160,0.100902,0.075336,7.088967
min,0.000000,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.200000,29.333333
25%,0.200000,1.0,0.460753,0.000000,0.542827,0.794415,0.000000,0.604330,0.743164,0.500000,0.000000,0.928246,43.844863
50%,0.297998,1.0,0.567203,0.206608,0.674291,0.901623,0.141886,0.804393,0.866887,0.606255,0.051855,1.000000,47.804199
75%,0.378048,1.0,0.667783,0.359734,0.818440,0.965009,0.674314,0.936720,0.934700,0.735576,0.124951,1.000000,53.436508
max,1.000000,1.0,1.000000,0.879029,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.600000,1.000000,74.750000


#### Definición de Listas de Columnas

Estas celdas definen listas de nombres de columnas para diferentes categorías de variables (miembros del hogar, educación). Es posible que estas listas fueran usadas en iteraciones previas del código o para propósitos de documentación. Si no se usan activamente en el pipeline actual, pueden ser consideradas como un remanente o para referencias futuras.

In [56]:
columnas_miembro  = ['es_jefe', 'ind_miembro_hogar', 'es_mujer', 'edad', 'en_pareja', 'trabajo_ingresos', 'factor_poblacion']

#### Re-Inspección de Columnas de `enaho_400_final`

Esta celda muestra las columnas del DataFrame `enaho_400_final`. Es probable que se haya incluido para verificar el estado de las columnas después de las transformaciones y antes de las fusiones finales. Si esta verificación ya está cubierta o no es crítica en este punto, puede ser un remanente.

In [57]:
columnas_educacion = ['edu_superior', 'edu_basica', 'alfabetizacion_funcional', 'lengua_materna_nativa', 'tic_uso_internet', 'tic_celular_propio', 'factor_educacion']

In [58]:
enaho_400_final.columns

Index(['anio', 'mes', 'conglome', 'vivienda', 'hogar', 'cod_persona', 'ubigeo',
       'dominio', 'estrato', 'es_jefe', 'es_mujer', 'edad', 'en_pareja',
       'salud_enfermedad_cronica', 'factor_salud', 'tiene_seguro',
       'tiene_limitacion'],
      dtype='str')

In [59]:
llaves_individuo = ["anio", "mes", "conglome", "vivienda", "hogar", "ubigeo", "dominio", "estrato"]
llaves_persona   = ["anio", "mes", "conglome", "vivienda", "hogar", "ubigeo", "dominio", "estrato", "cod_persona"]

df_personas = pd.merge(enaho_200_final, enaho_300_final, on=llaves_persona, how="inner")

df_personas = pd.merge(df_personas, enaho_400_final, on=llaves_persona, how="inner")

In [60]:
enaho_200_final.shape

(115145, 15)

In [61]:
df_personas.shape

(42228, 30)

In [62]:
enaho_200_final.columns

Index(['anio', 'mes', 'conglome', 'vivienda', 'hogar', 'cod_persona', 'ubigeo',
       'dominio', 'estrato', 'es_jefe', 'ind_miembro_hogar', 'es_mujer',
       'edad', 'en_pareja', 'factor_poblacion'],
      dtype='str')

In [63]:
df_personas[['factor_poblacion', 'factor_educacion', 'factor_salud']].describe()

,factor_poblacion,factor_educacion,factor_salud
count,42228.000000,42228.000000,42228.000000
mean,247.957943,243.368515,247.957943
std,187.253714,219.021486,187.253714
min,1.882153,0.192009,1.882153
25%,136.325531,99.086227,136.325531
50%,209.929977,185.789200,209.929977
75%,321.794647,320.651428,321.794647
max,1236.256958,3748.197266,1236.256958


In [64]:
import numpy as np
import pandas as pd

# =====================================================================
# 1. FUSIÓN DE LOS MÓDULOS A NIVEL DE INDIVIDUO (MERGE)
# =====================================================================

# Llaves de combinación por hogar e individuo
llaves_individuo = ["anio", "mes", "conglome", "vivienda", "hogar", "ubigeo"]
llaves_persona   = ["anio", "mes", "conglome", "vivienda", "hogar", "ubigeo", "cod_persona"]

# Módulo 200 + Módulo 300 (nivel persona)
df_personas = pd.merge(enaho_200_final, enaho_300_final, on=llaves_persona, how="inner")

# + Módulo 400 (salud) — excluir columnas ya presentes en 200 para evitar duplicados _x/_y
cols_salud_clean = [
    c
    for c in enaho_400_final.columns
    if c not in ["dominio", "estrato", "factor_salud"] or c in llaves_persona
]
df_personas = pd.merge(
    df_personas, enaho_400_final[cols_salud_clean], on=llaves_persona, how="inner"
)

# + Módulo 100 (vivienda/hogar) — se une por llave de hogar (sin cod_persona)
enaho_individual = pd.merge(
    df_personas, enaho_100_final, on=llaves_individuo, how="inner"
)

print(f"✅ Fusión completada. Registros individuales consolidados: {enaho_individual.shape}")


# =====================================================================
# 2. COLAPSO DE COLUMNAS DUPLICADAS (_x / _y) TRAS EL MERGE
# =====================================================================
def _coalesce_column_variants(df, base_name):
    variants = [c for c in df.columns if c == base_name or c.startswith(f"{base_name}_")]
    if not variants:
        return df
    if base_name in df.columns and len(variants) == 1:
        return df
    df = df.copy()
    df[base_name] = df[variants].bfill(axis=1).iloc[:, 0]
    drop_cols = [c for c in variants if c != base_name]
    if drop_cols:
        df = df.drop(columns=drop_cols)
    return df

for col_base in ["es_jefe", "es_mujer", "edad", "en_pareja"]:
    enaho_individual = _coalesce_column_variants(enaho_individual, col_base)

if "es_jefe" not in enaho_individual.columns and "P203" in enaho_individual.columns:
    enaho_individual = enaho_individual.rename(columns={"P203": "es_jefe"})


# =====================================================================
# 3. DEFINICIÓN DE VARIABLES PARA LA AGREGACIÓN POR DISTRITO
# =====================================================================
variables_a_agregar = [
    # Módulo 100 — Vivienda y servicios del hogar
    "viv_tipo",
    "viv_paredes",
    "viv_pisos",
    "viv_techos",
    "viv_hab_total",
    "viv_hab_dormir",
    "tic_celular",
    "tic_internet",
    "agua_todos_dias",
    "serv_electricidad",
    "nbi_vivienda",
    "nbi_hacinamiento",
    "nbi_sshh",
    "nbi_educacion",
    "nbi_dependencia",
    # Módulo 200 — Características del individuo
    "es_jefe",
    "es_mujer",
    "edad",
    "en_pareja",
    "trabajo_ingresos",
    # Módulo 300 — Educación y acceso digital
    "edu_superior",
    "edu_basica",
    "tic_uso_internet",
    "tic_celular_propio",
    # Módulo 400 — Salud y cobertura
    "salud_enfermedad_cronica",
    "lim_discapacidad_motriz",
    "lim_discapacidad_visual",
    "lim_discapacidad_habla",
    "lim_discapacidad_auditiva",
    "lim_discapacidad_comprension",
    "lim_discapacidad_relacion",
    "seg_essalud",
    "seg_privado",
    "seg_sis",
]

enaho_individual = enaho_individual[(enaho_individual.edad >= 18) & (enaho_individual.es_mujer == 1.0)]

✅ Fusión completada. Registros individuales consolidados: (104446, 44)


#### Filtrado de Mujeres Adultas

Después de fusionar los módulos y procesar las variables a nivel individual, se aplica un filtro para considerar únicamente a las **mujeres adultas** (edad igual o mayor a 18 años y `es_mujer` igual a 1.0). Este subconjunto de la población es el foco del análisis distrital.

In [65]:

# =====================================================================
# 4. AGREGACIÓN PONDERADA POR DISTRITO (ubigeo)
# =====================================================================
def _agregar_distrito_ponderado(group, variables_to_process):
    """Calcula la media ponderada por factor de expansión para cada variable."""
    pesos = group["factor_poblacion"]
    resultados = {}

    for col in variables_to_process:
        mascara_valida = group[col].notna()
        if not mascara_valida.any():
            resultados[col] = np.nan
            continue
        valores_validos = group.loc[mascara_valida, col]
        pesos_validos   = pesos.loc[mascara_valida]
        suma_pesos      = pesos_validos.sum()
        resultados[col] = (
            (valores_validos * pesos_validos).sum() / suma_pesos
            if suma_pesos > 0
            else np.nan
        )
    return pd.Series(resultados)


# Filtrar solo las variables que existen en el dataframe
variables_disponibles = [c for c in variables_a_agregar if c in enaho_individual.columns]

# Detectar columna de ponderación (cualquier columna que contenga 'factor')
candidatos_peso = [c for c in enaho_individual.columns if "factor" in c.lower()]
columna_peso = candidatos_peso[0] if candidatos_peso else None

def _collapse_group(group):
    if not variables_disponibles:
        return pd.Series(dtype=float)
    if columna_peso and columna_peso in group.columns:
        sub = group[variables_disponibles + [columna_peso]].rename(
            columns={columna_peso: "factor_poblacion"}
        )
    else:
        sub = group[variables_disponibles].copy()
        sub["factor_poblacion"] = 1.0
    return _agregar_distrito_ponderado(sub, variables_disponibles)

enaho_por_distrito = (
    enaho_individual.groupby("ubigeo")
    .apply(_collapse_group, include_groups=False)
    .reset_index()
)

print("Peso usado:", columna_peso if columna_peso else "ninguno (pesos iguales)")
print(f"Variables agregadas: {variables_disponibles}")
print(f"✅ Dataset final por distrito: {enaho_por_distrito.shape}")

Peso usado: factor_poblacion
Variables agregadas: ['viv_tipo', 'tic_celular', 'tic_internet', 'agua_todos_dias', 'serv_electricidad', 'nbi_vivienda', 'nbi_hacinamiento', 'nbi_sshh', 'nbi_educacion', 'nbi_dependencia', 'es_jefe', 'es_mujer', 'edad', 'en_pareja', 'edu_superior', 'edu_basica', 'tic_uso_internet', 'tic_celular_propio', 'salud_enfermedad_cronica']
✅ Dataset final por distrito: (1271, 20)


#### Filtrado de Confiabilidad Muestral por Distrito

Para asegurar la robustez de los indicadores distritales, se aplica un filtro basado en umbrales mínimos de observaciones muestrales. Se calculan el número de hogares, personas y mujeres por distrito. Solo aquellos distritos que cumplen con los umbrales definidos (`UMBRAL_HOGARES`, `UMBRAL_PERSONAS`, `UMBRAL_MUJERES`) son retenidos para el análisis final. Esto ayuda a evitar la publicación de indicadores basados en muestras muy pequeñas, que podrían no ser representativos.

In [66]:
# =====================================================================
# 5. FILTRO DE CONFIABILIDAD MUESTRAL POR DISTRITO
# =====================================================================

UMBRAL_HOGARES  = 20   # mínimo de hogares distintos encuestados por distrito
UMBRAL_PERSONAS = 30   # mínimo de personas distintas encuestadas por distrito
UMBRAL_MUJERES  = 10   # mínimo de mujeres (para variables de género)

# Calcular tamaños muestrales reales (sin ponderar) por distrito
conteo_hogares = (
    enaho_individual
    .drop_duplicates(subset=["ubigeo", "conglome", "vivienda", "hogar"])
    .groupby("ubigeo")
    .size()
    .rename("n_hogares")
)

conteo_personas = (
    enaho_individual
    .groupby("ubigeo")
    .size()
    .rename("n_personas")
)

conteo_mujeres = (
    enaho_individual[enaho_individual["es_mujer"] == 1.0]
    .groupby("ubigeo")
    .size()
    .rename("n_mujeres")
)

# Unir conteos al dataset distrital
enaho_por_distrito = (
    enaho_por_distrito
    .join(conteo_hogares,  on="ubigeo")
    .join(conteo_personas, on="ubigeo")
    .join(conteo_mujeres,  on="ubigeo")
)

# Rellenar NaN en n_mujeres (distritos sin mujeres en muestra)
enaho_por_distrito["n_mujeres"] = enaho_por_distrito["n_mujeres"].fillna(0).astype(int)

# Aplicar filtro
mascara_confiable = (
    (enaho_por_distrito["n_hogares"]  >= UMBRAL_HOGARES)  &
    (enaho_por_distrito["n_personas"] >= UMBRAL_PERSONAS) &
    (enaho_por_distrito["n_mujeres"]  >= UMBRAL_MUJERES)
)

enaho_distritos_confiables = enaho_por_distrito[mascara_confiable].copy()

n_total    = len(enaho_por_distrito)
n_retenido = len(enaho_distritos_confiables)

print(f"Distritos totales:    {n_total}")
print(f"Distritos retenidos:  {n_retenido}  ({n_retenido/n_total:.1%})")
print(f"Distritos descartados: {n_total - n_retenido}")
print(f"\nRango n_hogares:  {enaho_por_distrito['n_hogares'].min()}–{enaho_por_distrito['n_hogares'].max()}")
print(f"Rango n_personas: {enaho_por_distrito['n_personas'].min()}–{enaho_por_distrito['n_personas'].max()}")

Distritos totales:    1271
Distritos retenidos:  301  (23.7%)
Distritos descartados: 970

Rango n_hogares:  1–386
Rango n_personas: 1–598


#### Resumen Descriptivo de Distritos Confiables

Se genera un resumen descriptivo (`.describe()`) del DataFrame `enaho_distritos_confiables`. Esto proporciona estadísticas clave (media, desviación estándar, mínimos, máximos, cuartiles) para cada una de las variables agregadas a nivel distrital, permitiendo una visión general de la distribución de los indicadores en los distritos que cumplen con los criterios de confiabilidad.

In [67]:
enaho_distritos_confiables.describe()

,viv_tipo,tic_celular,tic_internet,agua_todos_dias,serv_electricidad,nbi_vivienda,nbi_hacinamiento,nbi_sshh,nbi_educacion,nbi_dependencia,...,edad,en_pareja,edu_superior,edu_basica,tic_uso_internet,tic_celular_propio,salud_enfermedad_cronica,n_hogares,n_personas,n_mujeres
count,301.000000,301.000000,301.000000,301.000000,301.000000,301.000000,301.000000,301.000000,301.000000,301.000000,...,301.000000,301.000000,301.000000,301.000000,301.000000,301.000000,301.000000,301.000000,301.000000,301.000000
mean,0.947225,0.965379,0.939696,0.816979,0.954248,0.068357,0.046621,0.054776,0.005260,0.007806,...,45.838414,0.511309,0.323823,0.599174,0.785078,0.852387,0.589488,66.737542,93.803987,93.803987
std,0.063124,0.051238,0.062320,0.258286,0.111843,0.098554,0.055074,0.100078,0.015152,0.022557,...,4.031972,0.102103,0.158129,0.137859,0.142552,0.114158,0.116709,59.735424,88.938620,88.938620
min,0.709549,0.531983,0.531983,0.000000,0.089270,0.000000,0.000000,0.000000,0.000000,0.000000,...,34.380498,0.268733,0.000000,0.139895,0.314718,0.198126,0.149873,20.000000,30.000000,30.000000
25%,0.923335,0.953977,0.925144,0.770785,0.968314,0.000000,0.001546,0.000000,0.000000,0.000000,...,43.387345,0.442262,0.228428,0.529512,0.707595,0.826623,0.527932,31.000000,40.000000,40.000000
50%,0.969977,0.983056,0.953977,0.925596,0.996635,0.027775,0.032541,0.015358,0.000000,0.000000,...,45.744803,0.499131,0.315898,0.608219,0.803242,0.889077,0.593061,45.000000,60.000000,60.000000
75%,1.000000,0.999877,0.978292,1.000000,1.000000,0.092168,0.065648,0.059601,0.000000,0.000000,...,47.965896,0.571713,0.408369,0.685032,0.896831,0.922836,0.662090,76.000000,108.000000,108.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,0.545660,0.393874,0.645237,0.121328,0.175569,...,59.937573,0.850046,0.860105,0.924834,1.000000,1.000000,0.912136,386.000000,598.000000,598.000000


#### Eliminación de Columnas de Conteo

Después de aplicar el filtro de confiabilidad, las columnas auxiliares (`n_hogares`, `n_personas`, `n_mujeres`) utilizadas para el filtrado ya no son necesarias y se eliminan del DataFrame `enaho_distritos_confiables` para mantenerlo limpio y enfocado en los indicadores.

In [68]:
enaho_distritos_confiables = enaho_distritos_confiables.drop(columns=["n_hogares", "n_personas", "n_mujeres"])
print(f"✅ Dataset final de distritos confiables: {enaho_distritos_confiables.shape}")

✅ Dataset final de distritos confiables: (301, 20)


#### Guardado del Dataset Final

El DataFrame `enaho_distritos_confiables`, que contiene los indicadores agregados por distrito y filtrados por confiabilidad, se guarda en un archivo CSV (`enaho_por_distrito.csv`) dentro del directorio `../data/processed/`. Este archivo es el resultado final del pipeline y está listo para ser utilizado en análisis posteriores.

In [69]:
enaho_distritos_confiables.to_csv('../data/processed/enaho_por_distrito.csv', index=False)